# Hospital Data Cleaning & Transformation
**Module 2 — Data Cleaning & Transformation**

This notebook loads the raw hospital data, cleans it, and saves the cleaned result as a CSV file.

**Place this notebook in the same folder as `hospital_raw_data.csv`** — the paths below use filenames only, so everything reads/writes right next to the notebook.

**Steps covered:**
1. Load & inspect raw data
2. Remove duplicate records
3. Standardize department names & text fields
4. Handle missing patient data
5. Standardize date formats
6. Normalize healthcare indicators (rate/percentage columns)
7. Validate
8. Save cleaned CSV

## 1. Load Data

In [1]:
import pandas as pd
import numpy as np

RAW_PATH = "hospital_raw_data.csv"
OUT_PATH_CSV = "hospital_cleaned.csv"

df = pd.read_csv(RAW_PATH)
print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


Loaded 10000 rows, 50 columns


,Patient ID,Hospital ID,Hospital Name,Hospital Type,City,State,Department,Department ID,Doctor,Nurses,...,Transfer_From_Department,Transfer_To_Department,Transfer_Date,Number_of_Transfers,Dept_ICU_Bed_Capacity_Derived,Dept_Staff_Capacity_Derived,Admissions_Rate_%_Derived,Staff_Utilization_%_Derived,Bed_Occupancy_Rate_%,Beds Occupied
0,PAT00001,HOSP00016,C K Hospital,Private,Mumbai,Maharashtra,General Medicine,DEPT004,Dr. Arjun Sharma,12,...,General Medicine,NaN,NaN,0,98,598,0.0,43.0,0.2,144
1,PAT00002,HOSP00019,Cure Well Hospital,Private,Bengaluru,Karnataka,General Surgery,DEPT010,Dr. Sneha Rao,15,...,General Surgery,NaN,NaN,0,100,583,0.2,49.4,0.2,48
2,PAT00003,HOSP00004,Yashoda Hospital,Private,Hyderabad,Telangana,Pulmonology,DEPT005,Dr. Priya Reddy,20,...,Pulmonology,NaN,NaN,0,97,590,0.0,30.7,0.2,82
3,PAT00004,HOSP00017,Dharma Hospital,Private,Delhi,Delhi,Pulmonology,DEPT005,Dr. Rahul Verma,23,...,Pulmonology,Cardiology,9/17/2025,1,99,597,0.2,28.5,0.4,142
4,PAT00005,HOSP00017,Dharma Hospital,Private,Delhi,Delhi,ICU,DEPT009,Dr. Priya Reddy,34,...,ICU,NaN,NaN,0,100,576,0.0,20.1,0.6,117


## 2. Initial Data Quality Assessment
Check for duplicates, missing values, and inconsistent categories before cleaning.

In [2]:
missing = df.isnull().sum()
missing = missing[missing > 0]
print("Columns with missing values:")
print(missing)
print()
print("Total missing %:", round(df.isnull().sum().sum() / df.size * 100, 2), "%")
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate Patient IDs:", df.duplicated(subset=['Patient ID']).sum())


Columns with missing values:
Insurance Provider        2463
Transfer_To_Department    8240
Transfer_Date             8240
dtype: int64

Total missing %: 3.79 %
Exact duplicate rows: 0
Duplicate Patient IDs: 0


## 3. Remove Duplicate Records
Drop exact duplicate rows, and drop duplicate `Patient ID` entries (a patient admission record should be unique).

In [3]:
rows_before = len(df)

exact_dupes = df.duplicated().sum()
df = df.drop_duplicates()

id_dupes = df.duplicated(subset=["Patient ID"]).sum()
df = df.drop_duplicates(subset=["Patient ID"], keep="first")

print(f"Removed {exact_dupes} exact duplicate rows")
print(f"Removed {id_dupes} duplicate Patient ID rows")
print(f"Rows: {rows_before} -> {len(df)}")


Removed 0 exact duplicate rows
Removed 0 duplicate Patient ID rows
Rows: 10000 -> 10000


## 4. Standardize Text Fields & Department Names
Trim whitespace, normalize casing, and reconcile `Department` against a single canonical `Department ID` per department (guards against any stray mismatches).

In [4]:
text_cols = df.select_dtypes(include="object").columns.tolist()

# Trim whitespace and normalize empty-string variants to NaN
for col in text_cols:
    df[col] = df[col].astype(str).str.strip()
    df.loc[df[col].isin(["nan", "None", ""]), col] = np.nan

# Title-case categorical/location fields (skip fields with acronyms like
# Insurance Provider "ICICI Lombard" or Doctor "Dr. Name", which are
# already well-formatted and would be broken by title-casing)
for col in ["Department", "Transfer_From_Department", "Transfer_To_Department",
            "Hospital Type", "City", "State",
            "Diagnosis", "Treatment", "Medication", "Equipment"]:
    df[col] = df[col].str.title()

print("Unique departments after standardization:")
print(sorted(df["Department"].dropna().unique()))


C:\Users\nagar\AppData\Local\Temp\ipykernel_16324\377349910.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include="object").columns.tolist()


Unique departments after standardization:
['Cardiology', 'General Medicine', 'General Surgery', 'Icu', 'Nephrology', 'Neurology', 'Oncology', 'Orthopedics', 'Psychiatry', 'Pulmonology']


In [5]:
# Reconcile Department -> Department ID using the most common mapping
dept_id_map = (
    df.dropna(subset=["Department", "Department ID"])
      .groupby("Department")["Department ID"]
      .agg(lambda s: s.mode().iloc[0])
      .to_dict()
)
df["Department ID"] = df["Department"].map(dept_id_map).fillna(df["Department ID"])
df[["Department", "Department ID"]].drop_duplicates().sort_values("Department ID")


,Department,Department ID
11,Cardiology,DEPT001
8,Neurology,DEPT002
10,Orthopedics,DEPT003
0,General Medicine,DEPT004
2,Pulmonology,DEPT005
20,Nephrology,DEPT006
5,Oncology,DEPT007
25,Psychiatry,DEPT008
4,Icu,DEPT009
1,General Surgery,DEPT010


## 5. Handle Missing Patient Data

- **`Insurance Provider`** missing → the patient is self-paying, so we fill with `"Self-Pay"` rather than dropping the record.
- **`Transfer_To_Department`** / **`Transfer_Date`** missing → these are only populated when `Transferred == "Yes"`. A missing value here is structurally expected (no transfer occurred), so we encode it explicitly as `"Not Transferred"` instead of leaving a blank.
- Any remaining missing **numeric** values are filled with the column median; any remaining missing **text** values are filled with `"Unknown"` as a safety net.

In [6]:
missing_before = df.isnull().sum().sum()
missing_pct_before = round(missing_before / df.size * 100, 2)

df["Insurance Provider"] = df["Insurance Provider"].fillna("Self-Pay")
df["Transfer_To_Department"] = df["Transfer_To_Department"].fillna("Not Transferred")
df["Transfer_Date"] = df["Transfer_Date"].fillna("Not Transferred")

num_cols = df.select_dtypes(include=np.number).columns
for col in num_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

df[text_cols] = df[text_cols].fillna("Unknown")

missing_after = df.isnull().sum().sum()
missing_pct_after = round(missing_after / df.size * 100, 2)
print(f"Missing values: {missing_before} ({missing_pct_before}%) -> {missing_after} ({missing_pct_after}%)")


Missing values: 18943 (3.79%) -> 0 (0.0%)


## 6. Standardize Date Formats
The raw data mixes `M/D/YYYY` and `YYYY-MM-DD HH:MM:SS` formats. Convert all date fields to a single ISO-8601 `YYYY-MM-DD` format for consistent Tableau/Excel parsing.

In [7]:
for col in ["Admission Date", "Discharge Date"]:
    df[col] = pd.to_datetime(df[col], format="mixed", errors="coerce").dt.strftime("%Y-%m-%d")

# Transfer_Date mixes real dates with the "Not Transferred" placeholder
mask = df["Transfer_Date"] != "Not Transferred"
df.loc[mask, "Transfer_Date"] = pd.to_datetime(
    df.loc[mask, "Transfer_Date"], format="mixed", errors="coerce"
).dt.strftime("%Y-%m-%d")

df[["Admission Date", "Discharge Date", "Transfer_Date"]].head()


,Admission Date,Discharge Date,Transfer_Date
0,2026-02-13,2026-02-27,Not Transferred
1,2026-05-16,2026-05-20,Not Transferred
2,2026-01-04,2026-05-04,Not Transferred
3,2025-09-13,2025-09-24,2025-09-17
4,2025-09-11,2025-12-11,Not Transferred


## 7. Normalize Healthcare Indicators
`Admissions_Rate_%_Derived` and `Bed_Occupancy_Rate_%` were stored as raw fractions (range ~0.0–1.4), while `Staff_Utilization_%_Derived` was already a true 0–100 percentage. Rescale the fraction columns ×100 so all rate indicators share one consistent percentage scale — required for meaningful comparison and reporting.

In [8]:
print("Before normalization:")
print(df[["Admissions_Rate_%_Derived", "Bed_Occupancy_Rate_%", "Staff_Utilization_%_Derived"]].describe())

for col in ["Admissions_Rate_%_Derived", "Bed_Occupancy_Rate_%"]:
    df[col] = (df[col] * 100).round(2)
df["Staff_Utilization_%_Derived"] = df["Staff_Utilization_%_Derived"].round(2)

# Standardize Yes/No flag columns to consistent capitalization
for col in ["Bed Occupied", "Readmission", "Transferred"]:
    df[col] = df[col].str.capitalize()

print("\nAfter normalization:")
print(df[["Admissions_Rate_%_Derived", "Bed_Occupancy_Rate_%", "Staff_Utilization_%_Derived"]].describe())


Before normalization:
       Admissions_Rate_%_Derived  Bed_Occupancy_Rate_%  \
count               10000.000000           10000.00000   
mean                    0.110160               0.36757   
std                     0.109845               0.18417   
min                     0.000000               0.20000   
25%                     0.000000               0.20000   
50%                     0.200000               0.40000   
75%                     0.200000               0.40000   
max                     0.600000               1.40000   

       Staff_Utilization_%_Derived  
count                 10000.000000  
mean                     57.924860  
std                      25.296697  
min                      13.300000  
25%                      36.375000  
50%                      58.200000  
75%                      79.600000  
max                     100.000000  

After normalization:
       Admissions_Rate_%_Derived  Bed_Occupancy_Rate_%  \
count               10000.000000          

## 8. Final Validation

In [9]:
final_missing_pct = round(df.isnull().sum().sum() / df.size * 100, 2)
print(f"Final missing-value rate: {final_missing_pct}%  (target: < 2%)")
assert final_missing_pct < 2.0, "Missing value threshold not met!"

print(f"Final shape: {df.shape}")
print(f"Duplicate rows remaining: {df.duplicated().sum()}")
print(f"Duplicate Patient IDs remaining: {df.duplicated(subset=['Patient ID']).sum()}")


Final missing-value rate: 0.0%  (target: < 2%)
Final shape: (10000, 50)
Duplicate rows remaining: 0
Duplicate Patient IDs remaining: 0


## 9. Save Cleaned CSV
Save the cleaned `df` to `hospital_cleaned.csv`.

In [10]:
df.to_csv(OUT_PATH_CSV, index=False)
print(f"Saved cleaned dataset to {OUT_PATH_CSV}")
print(f"Final shape: {df.shape}")
df.head()


Saved cleaned dataset to hospital_cleaned.csv
Final shape: (10000, 50)


,Patient ID,Hospital ID,Hospital Name,Hospital Type,City,State,Department,Department ID,Doctor,Nurses,...,Transfer_From_Department,Transfer_To_Department,Transfer_Date,Number_of_Transfers,Dept_ICU_Bed_Capacity_Derived,Dept_Staff_Capacity_Derived,Admissions_Rate_%_Derived,Staff_Utilization_%_Derived,Bed_Occupancy_Rate_%,Beds Occupied
0,PAT00001,HOSP00016,C K Hospital,Private,Mumbai,Maharashtra,General Medicine,DEPT004,Dr. Arjun Sharma,12,...,General Medicine,Not Transferred,Not Transferred,0,98,598,0.0,43.0,20.0,144
1,PAT00002,HOSP00019,Cure Well Hospital,Private,Bengaluru,Karnataka,General Surgery,DEPT010,Dr. Sneha Rao,15,...,General Surgery,Not Transferred,Not Transferred,0,100,583,20.0,49.4,20.0,48
2,PAT00003,HOSP00004,Yashoda Hospital,Private,Hyderabad,Telangana,Pulmonology,DEPT005,Dr. Priya Reddy,20,...,Pulmonology,Not Transferred,Not Transferred,0,97,590,0.0,30.7,20.0,82
3,PAT00004,HOSP00017,Dharma Hospital,Private,Delhi,Delhi,Pulmonology,DEPT005,Dr. Rahul Verma,23,...,Pulmonology,Cardiology,2025-09-17,1,99,597,20.0,28.5,40.0,142
4,PAT00005,HOSP00017,Dharma Hospital,Private,Delhi,Delhi,Icu,DEPT009,Dr. Priya Reddy,34,...,Icu,Not Transferred,Not Transferred,0,100,576,0.0,20.1,60.0,117
